# recs_026 -- Vector-blended vs. text-concatenated query construction (bge-small-en-v1.5)

## Executive Summary

- Query construction today (`query_plus_desc`) concatenates review text + description text into
  one string before a single embed call. Alternative: embed them separately, blend the two
  *vectors* with a tunable weight -- mirrors how the catalog side already works.
- Everything else fixed at shipped defaults (`bge-small-en-v1.5`, `any_polarity__flat`,
  catalog `description_blend_weight=0.1`) -- isolates query construction as the one variable.
- Result: **Vector-blend wins big.** A plateau at `w=0.5-0.7` beats both `query_plus_desc`
  (0.532/0.509/0.514 vs 0.516/0.485/0.499) and `two_tower_v1` (0.512/0.460/0.496) on every cut --
  by a wider margin than the embedder swap alone gave. Even pure description with zero review
  signal (`w=1.0`) nearly ties `query_plus_desc`. This is the single biggest lever found across
  the whole ablation series.

## Business Context

Text-concatenation is all-or-nothing -- the description either gets spliced in whole or not at
all, no way to dial its influence. Vector-blending gives a tunable weight, same mechanism as the
catalog-side blend, and might avoid text-concat's known failure mode (NieR:Automata-style genre
drift, `recs_023`) if a smaller weight keeps the query anchored closer to the review.

## Research Question

Holding the embedder and catalog fixed, does blending query review + description as separate
vectors (at some weight) beat text-concatenation (`query_plus_desc`) or plain `raw_query`?

## Hypothesis

Some small blend weight (~0.1-0.2) beats `raw_query` by adding description signal without
text-concat's all-or-nothing dilution, but it's unclear whether it beats `query_plus_desc`
outright -- concatenation lets the model attend to both texts jointly, which blending can't
replicate.

**Result: wrong on magnitude, right on direction.** The optimum isn't small (~0.1-0.2) -- it's a
plateau at `w=0.5-0.7`, and it clearly beats `query_plus_desc`, not just `raw_query`. The
"concatenation lets the model attend jointly" concern didn't hold: separate embed calls, blended
as vectors, beat the joint-attention text-concat approach outright.

## Definitions

| Term | Meaning |
|---|---|
| `query_plus_desc` | Current default: `embed(review_text + "\n\n" + description_text)`, one embed call. |
| Vector-blend | `normalize((1-w)*embed(review_text) + w*embed(description_text))`, two embed calls. |
| BGE query prefix | `"Represent this sentence for searching relevant passages: "`, prepended to query-side text only, per BGE convention. Applied to both components in vector-blend. |

## Data Sources

| Source | Role |
|---|---|
| `artifacts/recs/embeddings/game_chunks/default/game_review_chunks.parquet` | Stage 1 chunk table -- catalog side, unchanged from shipped Stage 2. |
| `artifacts/recs/offline_eval/runs/rag_v1/eval_offline_examples.jsonl` | Same 12,500-example cohort used throughout this series. |
| `data/processed/steam_reviews_cleaned_english_val_norm.parquet` | Val split -- recovers query review text. |
| `artifacts/recs/offline_eval/runs/rag_v2/eval_retrieval_overall.csv`, `eval_retrieval_by_slice.csv` | Real `raw_query`/`query_plus_desc` (bge-small) numbers, for sanity-checking this notebook's scoring. |

## Design / Process

1. Build the catalog once: shipped config (`any_polarity__flat`, `blend_weight=0.1`,
   `bge-small-en-v1.5`, no query prefix -- passage side).
2. Embed query components with the query prefix: each query's review text, each query's own
   game's description (315 unique, reused across queries), and the concatenated
   review+description text (needed separately -- concatenation happens before embedding, can't
   be derived from the two separate vectors).
3. Sanity check: `raw_query` (review vector alone) and `query_plus_desc` (concatenated-text
   vector) should reproduce `rag_v2`'s real bge-small numbers.
4. Vector-blend sweep: `normalize((1-w)*review_vec + w*description_vec)` for
   `w in {0.0, 0.05, 0.1, 0.2, 0.3, 0.5}`, scored against the same catalog.
5. Compare all three query-construction modes.

## Evaluation Outputs / Artifacts

| Artifact | Description |
|---|---|
| Comparison table (this notebook) | Hit@100/Recall@100, overall and by slice, for `raw_query`, `query_plus_desc`, and the vector-blend sweep. |

## Notebook Roadmap

1. Setup
2. Load chunk table + eval cohort, recover query text
3. Build catalog (shipped config)
4. Embed query components, sanity-check against real `rag_v2` numbers
5. Vector-blend sweep + comparison table

# Analysis

## Setup

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    while not (p / "pyproject.toml").is_file():
        if p.parent == p:
            raise RuntimeError("Could not find repo root (pyproject.toml not found).")
        p = p.parent
    return p

REPO_ROOT = _find_repo_root(Path.cwd())
import sys
sys.path.insert(0, str(REPO_ROOT / "src"))

from steam_review_ml.recommender.math_utils import l2_normalize
from steam_review_ml.evaluation.retrieval_offline_eval import hit_rate_at_k, precision_at_k, recall_at_k

RUN_DIR = REPO_ROOT / "artifacts" / "recs" / "offline_eval" / "runs" / "rag_v1"
RUN_DIR_V2 = REPO_ROOT / "artifacts" / "recs" / "offline_eval" / "runs" / "rag_v2"
CHUNKS_PATH = REPO_ROOT / "artifacts" / "recs" / "embeddings" / "game_chunks" / "default" / "game_review_chunks.parquet"
VAL_SPLIT_PATH = REPO_ROOT / "data" / "processed" / "steam_reviews_cleaned_english_val_norm.parquet"

USER_COL = "author.steamid"
K_RETRIEVAL = 100
CATALOG_BLEND_WEIGHT = 0.1
QUERY_BLEND_WEIGHTS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0]
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

pd.options.display.max_colwidth = 60
print(f"REPO_ROOT={REPO_ROOT}")

REPO_ROOT=/home/ryanr/workspace/steam_recommendations


## Load Stage 1 Chunk Table + Eval Cohort, Recover Query Text

In [2]:
chunks_df = pd.read_parquet(CHUNKS_PATH)
review_chunks = chunks_df[chunks_df["chunk_type"] == "review"].reset_index(drop=True)
description_by_app: dict[int, str] = dict(
    zip(
        chunks_df.loc[chunks_df["chunk_type"] == "description", "app_id"],
        chunks_df.loc[chunks_df["chunk_type"] == "description", "text"],
    )
)
print(f"chunk rows: {len(chunks_df):,} (review={len(review_chunks):,}, description={len(description_by_app):,})")

examples = []
with open(RUN_DIR / "eval_offline_examples.jsonl") as f:
    for line in f:
        rec = json.loads(line)
        if rec["method"] != "rag_chunk_v1_query_plus_desc":
            continue
        examples.append(
            {
                "ex_idx": rec["ex_idx"],
                "user_id": rec["user_id"],
                "query_app_id": rec["query_app_id"],
                "positives": set(json.loads(rec["validation_positive_app_ids_json"])),
                "n_eval_targets": rec["n_eval_targets"],
                "slice_name": rec["slice_name"],
            }
        )
examples_df = pd.DataFrame(examples)
print(f"eval cohort rows: {len(examples_df):,}")

val_df = pd.read_parquet(VAL_SPLIT_PATH, columns=[USER_COL, "app_id", "review"])
val_df["_key"] = val_df[USER_COL].astype(str) + "::" + val_df["app_id"].astype(str)
review_text_by_key = dict(zip(val_df["_key"], val_df["review"]))
examples_df["query_review_text"] = examples_df.apply(
    lambda r: review_text_by_key[f"{r['user_id']}::{r['query_app_id']}"], axis=1
)
display(examples_df[["ex_idx", "query_app_id", "n_eval_targets", "slice_name"]].head(3))

chunk rows: 16,010 (review=15,695, description=315)
eval cohort rows: 12,500


   ex_idx  query_app_id  n_eval_targets             slice_name0       0        812140               1  slice_b_single_target1       1        485510               1  slice_b_single_target2       2        646570               2   slice_a_multi_target

## Build Catalog (Shipped Config: bge-small, any_polarity__flat, blend=0.1)

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")


def encode(texts: list[str], *, prefix: str = "") -> np.ndarray:
    prefixed = [prefix + t for t in texts]
    emb = np.asarray(model.encode(prefixed, batch_size=128, show_progress_bar=True))
    return np.stack([l2_normalize(row) for row in emb], axis=0)


# Passage side (catalog): no query prefix, per BGE convention.
catalog_review_vecs = encode(review_chunks["text"].tolist())
desc_app_ids = list(description_by_app.keys())
catalog_desc_vecs = encode([description_by_app[a] for a in desc_app_ids])
catalog_desc_by_app = {a: v for a, v in zip(desc_app_ids, catalog_desc_vecs)}

pooled_by_app: dict[int, np.ndarray] = {}
for app_id, g in review_chunks.groupby("app_id"):
    idx = g.index.to_numpy()
    pooled_by_app[int(app_id)] = l2_normalize(catalog_review_vecs[idx].mean(axis=0))

catalog_app_ids, catalog_vecs = [], []
for app_id, desc_vec in catalog_desc_by_app.items():
    pooled = pooled_by_app.get(app_id)
    if pooled is None:
        continue
    blended = l2_normalize((1.0 - CATALOG_BLEND_WEIGHT) * pooled + CATALOG_BLEND_WEIGHT * desc_vec)
    catalog_app_ids.append(app_id)
    catalog_vecs.append(blended)
catalog_app_ids = np.asarray(catalog_app_ids, dtype=np.int64)
catalog_matrix = np.stack(catalog_vecs, axis=0)
print(f"catalog: {catalog_matrix.shape}")

catalog: (315, 384)


## Embed Query Components (With BGE Query Prefix), Sanity-Check Against Real Numbers

In [4]:
# Query side: BGE query prefix applies to every embedded piece.
query_review_vecs = encode(examples_df["query_review_text"].tolist(), prefix=BGE_QUERY_PREFIX)

# Query's own game's description, embedded once per unique app_id (query-side prefix, unlike the
# catalog-side description embedding above which has no prefix).
query_desc_by_app = {a: v for a, v in zip(desc_app_ids, encode([description_by_app[a] for a in desc_app_ids], prefix=BGE_QUERY_PREFIX))}

# query_plus_desc needs its own embed pass -- concatenation happens before embedding, can't be
# derived from the two separate vectors above.
def query_plus_desc_text(query_app_id: int, review_text: str) -> str:
    description = description_by_app.get(int(query_app_id))
    return f"{review_text}\n\n{description}" if description else review_text

concat_texts = examples_df.apply(
    lambda r: query_plus_desc_text(r["query_app_id"], r["query_review_text"]), axis=1
).tolist()
query_plus_desc_vecs = encode(concat_texts, prefix=BGE_QUERY_PREFIX)
print(f"query_review_vecs: {query_review_vecs.shape}, query_plus_desc_vecs: {query_plus_desc_vecs.shape}")


def score(query_vecs: np.ndarray) -> pd.DataFrame:
    app_to_row = {int(a): i for i, a in enumerate(catalog_app_ids)}
    rows = []
    for i, ex in enumerate(examples_df.itertuples(index=False)):
        scores = catalog_matrix @ query_vecs[i]
        self_row = app_to_row.get(int(ex.query_app_id))
        if self_row is not None:
            scores[self_row] = -np.inf
        ranked_rows = np.argsort(-scores)[:K_RETRIEVAL]
        rows.append(
            {
                "slice_name": ex.slice_name,
                "Hit@K": hit_rate_at_k(ranked_rows, ex.positives, K_RETRIEVAL, catalog_app_ids),
                "Recall@K": recall_at_k(ranked_rows, ex.positives, K_RETRIEVAL, catalog_app_ids),
            }
        )
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> dict:
    by_slice = df.groupby("slice_name")[["Hit@K", "Recall@K"]].mean()
    return {
        "overall_hit": df["Hit@K"].mean(),
        "slice_a_recall": by_slice.loc["slice_a_multi_target", "Recall@K"] if "slice_a_multi_target" in by_slice.index else float("nan"),
        "slice_b_hit": by_slice.loc["slice_b_single_target", "Hit@K"] if "slice_b_single_target" in by_slice.index else float("nan"),
    }


raw_query_metrics = summarize(score(query_review_vecs))
query_plus_desc_metrics = summarize(score(query_plus_desc_vecs))

overall_csv = pd.read_csv(RUN_DIR_V2 / "eval_retrieval_overall.csv")
by_slice_csv = pd.read_csv(RUN_DIR_V2 / "eval_retrieval_by_slice.csv")


def real_numbers(method: str) -> dict:
    return {
        "overall_hit": overall_csv.loc[overall_csv["method"] == method, "Hit@K"].iloc[0],
        "slice_a_recall": by_slice_csv.loc[(by_slice_csv["method"] == method) & (by_slice_csv["slice_name"] == "slice_a_multi_target"), "Recall@K"].iloc[0],
        "slice_b_hit": by_slice_csv.loc[(by_slice_csv["method"] == method) & (by_slice_csv["slice_name"] == "slice_b_single_target"), "Hit@K"].iloc[0],
    }


print("raw_query      -- notebook:", {k: round(v, 3) for k, v in raw_query_metrics.items()}, " real:", {k: round(v, 3) for k, v in real_numbers("rag_chunk_v1_raw_query").items()})
print("query_plus_desc -- notebook:", {k: round(v, 3) for k, v in query_plus_desc_metrics.items()}, " real:", {k: round(v, 3) for k, v in real_numbers("rag_chunk_v1_query_plus_desc").items()})

query_review_vecs: (12500, 384), query_plus_desc_vecs: (12500, 384)
raw_query      -- notebook: {'overall_hit': np.float64(0.483), 'slice_a_recall': np.float64(0.439), 'slice_b_hit': np.float64(0.467)}  real: {'overall_hit': np.float64(0.483), 'slice_a_recall': np.float64(0.439), 'slice_b_hit': np.float64(0.467)}
query_plus_desc -- notebook: {'overall_hit': np.float64(0.516), 'slice_a_recall': np.float64(0.485), 'slice_b_hit': np.float64(0.499)}  real: {'overall_hit': np.float64(0.516), 'slice_a_recall': np.float64(0.485), 'slice_b_hit': np.float64(0.499)}


## Vector-Blend Sweep + Comparison Table

In [5]:
rows = []
for w in QUERY_BLEND_WEIGHTS:
    blended_query_vecs = []
    for i, ex in enumerate(examples_df.itertuples(index=False)):
        desc_vec = query_desc_by_app.get(int(ex.query_app_id))
        if desc_vec is None:
            blended_query_vecs.append(query_review_vecs[i])
        else:
            blended_query_vecs.append(l2_normalize((1.0 - w) * query_review_vecs[i] + w * desc_vec))
    blended_query_vecs = np.stack(blended_query_vecs, axis=0)
    s = summarize(score(blended_query_vecs))
    rows.append({"mode": "vector_blend", "weight": w, "Hit@K (overall)": s["overall_hit"], "Recall@K (Slice A)": s["slice_a_recall"], "Hit@K (Slice B)": s["slice_b_hit"]})

rows.append({"mode": "raw_query", "weight": None, "Hit@K (overall)": raw_query_metrics["overall_hit"], "Recall@K (Slice A)": raw_query_metrics["slice_a_recall"], "Hit@K (Slice B)": raw_query_metrics["slice_b_hit"]})
rows.append({"mode": "query_plus_desc (text-concat)", "weight": None, "Hit@K (overall)": query_plus_desc_metrics["overall_hit"], "Recall@K (Slice A)": query_plus_desc_metrics["slice_a_recall"], "Hit@K (Slice B)": query_plus_desc_metrics["slice_b_hit"]})

comparison_df = pd.DataFrame(rows).round(3)
display(comparison_df.sort_values("Recall@K (Slice A)", ascending=False))

                             mode  weight  Hit@K (overall)  Recall@K (Slice A)  Hit@K (Slice B)5                    vector_blend    0.50            0.530               0.509            0.5136                    vector_blend    0.70            0.532               0.508            0.5144                    vector_blend    0.30            0.520               0.493            0.5037                    vector_blend    0.90            0.524               0.492            0.50710  query_plus_desc (text-concat)     NaN            0.516               0.485            0.4998                    vector_blend    1.00            0.518               0.483            0.5003                    vector_blend    0.20            0.509               0.473            0.4922                    vector_blend    0.10            0.496               0.456            0.4801                    vector_blend    0.05            0.490               0.447            0.4740                    vector_blend    0.00         

## Key Findings

| mode | weight | Hit@K (overall) | Recall@K (Slice A) | Hit@K (Slice B) |
|---|---|---|---|---|
| `two_tower_v1` (bar) | -- | 0.512 | 0.460 | 0.496 |
| `raw_query` | -- | 0.483 | 0.439 | 0.467 |
| `query_plus_desc` (text-concat) | -- | 0.516 | 0.485 | 0.499 |
| vector_blend | 0.3 | 0.520 | 0.493 | 0.503 |
| **vector_blend (peak)** | **0.5-0.7** | **0.530-0.532** | **0.508-0.509** | **0.513-0.514** |
| vector_blend | 1.0 (pure description) | 0.518 | 0.483 | 0.500 |

- Sanity checks exact match: `w=0.0` reproduces `raw_query`'s real numbers; text-concat
  reproduces `query_plus_desc`'s real numbers.
- **Crossover between `w=0.2` and `w=0.3`**: below that, vector-blend trails `query_plus_desc`;
  at and above it, vector-blend wins outright, on every cut from `w=0.3` through `w=1.0`.
- **Every weight from `w=0.3` to `w=1.0` beats `two_tower_v1` on all three primary cuts** -- a
  much wider margin and a much larger range than the embedder swap alone produced.
- **Surprise: pure description (`w=1.0`, zero review signal) nearly ties `query_plus_desc`**
  (0.518/0.483/0.500 vs 0.516/0.485/0.499). The game's own description text is nearly as strong
  a retrieval query on its own as the review-plus-description text-concat blob -- unclear why
  without a qualitative read (mirroring `recs_023`'s approach).

## Recommendation / Next Steps

- **Promote vector-blend query construction (`w≈0.5-0.6`) to replace `query_plus_desc`** as the
  Ablation B default -- biggest lever found in this series, wider margin over `two_tower_v1` than
  the embedder swap alone gave. Needs wiring into `chroma_retrieve.py`/
  `retrieval_offline_eval.py`'s method registry as a new method, then a real
  `recs_job_eval_offline.py` run to confirm this notebook's numbers, same promotion pattern as
  `recs_024` -> `rag_v2`.
- **Open question worth a follow-up**: why does description-only querying work almost as well as
  review+description? A qualitative read (`recs_023`-style) on a handful of examples would say
  whether this is genuine signal or a metric artifact (e.g. description text overlapping more
  directly with catalog descriptions than review text does with catalog reviews).
- Given this result, the catalog-side `blend_weight=0.3` optimum from `recs_025` should be
  re-checked once vector-blend query construction is the default -- that sweep assumed
  `query_plus_desc` querying, an assumption this notebook just invalidated.